In [1]:
from pathlib import Path
import json

references = {}

for path in Path("/home/souvikm/Downloads/test-elsevier/test-pipeline/doc_json").iterdir():
    if path.is_file() and path.suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            # print(data['file'])
            # print(f"Loaded {path.name}:")
            # print(data.keys())
            c = 0
            for d in data.keys():
                # print(data['file'])
                if 'References' in d :
                    references[data['file']] = data[d] or ""

print(len(references))

18


In [2]:
import journal_list_helper

path = journal_list_helper.download_medline_journals()

journals = journal_list_helper.parse_medline_journals(path)

journal_dict = journal_list_helper.create_journal_dict(journals)

print(len(journal_dict))
print(journal_dict.get("American Journal of Preventive Medicine".lower())['med_abbr'])

37937
Am J Prev Med


In [3]:
import reference_helper
print(len(references))
res_list = dict()
for k, ref in references.items():
    all_refs = ref.split('\n')
    all_ref_objs = []
    for r in all_refs:
        if 'reference' in r.lower():
            continue
        # print(r)
        print(reference_helper.parse_reference_deter(r))
        all_ref_objs.append(reference_helper.parse_reference_deter(r))
        # print()
    res_list[k] = all_ref_objs

print(len(res_list))

18
{'author': 'Kuypers KP, Ng L, Erritzoe D, et al.', 'title': 'Microdosing psychedelics: More questions than answers? An overview and suggestions for future research.', 'numbers': 'J Psychopharmacol. 2019;33(9):1039-1057.', 'doi': '10.1177/0269881119857204', 'actual_ref': '1.\tKuypers KP, Ng L, Erritzoe D, et al. Microdosing psychedelics: More questions than answers? An overview and suggestions for future research. J Psychopharmacol. 2019;33(9):1039-1057. doi:10.1177/0269881119857204'}
{'author': 'Fadiman J, Gruber J', 'title': 'Microdosing for Health, Healing, and Enhanced Performance. St. Martin’s Essentials; 2025.', 'numbers': '', 'doi': '', 'actual_ref': '2.\tFadiman J, Gruber J. Microdosing for Health, Healing, and Enhanced Performance. St. Martin’s Essentials; 2025.'}
{'author': 'Lea T, Amada N, Jungaberle H, Schecke H, Klein M', 'title': 'Microdosing psychedelics: Motivations, subjective effects and harm reduction.', 'numbers': 'International Journal of Drug Policy. 2020;75:102

In [5]:
reference_helper.parse_reference_deter('1.	Ndumele CE, Neeland IJ, Tuttle KR, Chow SL, Mathew RO, Khan SS, Coresh J, Baker-Smith CM, Carnethon MR, Després JP, et al. A Synopsis of the Evidence for the Science and Clinical Management of Cardiovascular-Kidney-Metabolic (CKM) Syndrome: A Scientific Statement From the American Heart Association. Circulation. 2023;148:1636-1664. doi: https://doi.org/10.1161/cir.0000000000001186.')

{'author': 'Ndumele CE, Neeland IJ, Tuttle KR, Chow SL, Mathew RO, Khan SS, Coresh J, Baker-Smith CM, Carnethon MR, Després JP, et al.',
 'title': 'A Synopsis of the Evidence for the Science and Clinical Management of Cardiovascular-Kidney-Metabolic (CKM) Syndrome: A Scientific Statement From the American Heart Association. Circulation. 2023;148:1636-1664. doi:',
 'numbers': '',
 'doi': '10.1161/cir.0000000000001186',
 'actual_ref': '1.\tNdumele CE, Neeland IJ, Tuttle KR, Chow SL, Mathew RO, Khan SS, Coresh J, Baker-Smith CM, Carnethon MR, Després JP, et al. A Synopsis of the Evidence for the Science and Clinical Management of Cardiovascular-Kidney-Metabolic (CKM) Syndrome: A Scientific Statement From the American Heart Association. Circulation. 2023;148:1636-1664. doi: https://doi.org/10.1161/cir.0000000000001186.'}

In [4]:
import reference_create_excel

final_ref_map = {}

for k,all_refs in res_list.items():
    final_ref_map[k] = reference_helper.get_ref_result_list(all_refs, journal_dict, 3)

Processing 3 references
--------ACTUAL REF1.	Kuypers KP, Ng L, Erritzoe D, et al. Microdosing psychedelics: More questions than answers? An overview and suggestions for future research. J Psychopharmacol. 2019;33(9):1039-1057. doi:10.1177/0269881119857204
found in dict
----------NUMBERS: {'year': '2019', 'volume': '33', 'issue': '9', 'pages': '1039-1057', 'article_number': ''}
---------NUMBERS TEXT: 2019;33(9):1039-1057
FINAL REF:
1. Kuypers KPC, Ng L, Erritzøe D, et al. Microdosing psychedelics: more questions than answers? An overview and suggestions for future research. J Psychopharmacol. 2019;33(9):1039-1057. https://doi.org/10.1177/0269881119857204.

--------ACTUAL REF2.	Fadiman J, Gruber J. Microdosing for Health, Healing, and Enhanced Performance. St. Martin’s Essentials; 2025.
----------NUMBERS: {'year': '', 'volume': '', 'issue': '', 'pages': '', 'article_number': ''}
---------NUMBERS TEXT: None
FINAL REF:
2. Microdosing for health, healing, and enhanced performance. St. Marti

In [ ]:
len(final_ref_map)

In [ ]:
final_ref_map

In [ ]:
import re
from pathlib import Path

from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter


def create_reference_edits_excel(data, output_path="reference_edits.xlsx"):
    """
    Create an Excel workbook from reference-edit JSON data.

    Parameters
    ----------
    data : dict
        Dictionary in the format:
        {
            "filename1.docx": [
                {
                    "original": "...",
                    "edited": "...",
                    "edits": {
                        "text_edits": [...],
                        "missing_components": [...],
                        "warnings": [...]
                    }
                }
            ],
            "filename2.docx": [...]
        }

    output_path : str
        Path where the Excel workbook will be saved.

    Returns
    -------
    str
        Path to the generated Excel workbook.
    """

    wb = Workbook()
    wb.remove(wb.active)

    # ---------------------------------------------------------
    # Styles
    # ---------------------------------------------------------

    header_font = Font(bold=True, color="FFFFFF")
    header_fill = PatternFill(
        fill_type="solid",
        fgColor="1F4E78"
    )

    thin_gray = Side(style="thin", color="D9E1F2")
    cell_border = Border(
        left=thin_gray,
        right=thin_gray,
        top=thin_gray,
        bottom=thin_gray
    )

    headers = [
        "Original",
        "Edited",
        "Edit Type",
        "Details"
    ]

    # ---------------------------------------------------------
    # Helper: create a valid, unique worksheet name
    # ---------------------------------------------------------

    used_sheet_names = set()

    def get_sheet_name(filename):
        # Excel prohibits: \ / * ? : [ ]
        name = re.sub(r'[\\/*?:\[\]]', "_", str(filename))
        name = name.strip("'") or "Sheet"

        # Excel worksheet names cannot exceed 31 characters.
        base_name = name[:31]
        sheet_name = base_name
        counter = 1

        while sheet_name.casefold() in used_sheet_names:
            suffix = f"_{counter}"
            sheet_name = base_name[:31 - len(suffix)] + suffix
            counter += 1

        used_sheet_names.add(sheet_name.casefold())
        return sheet_name

    # ---------------------------------------------------------
    # Create one worksheet for each file
    # ---------------------------------------------------------

    for filename, references in data.items():

        ws = wb.create_sheet(
            title=get_sheet_name(filename)
        )

        ws.append(headers)

        # Format header
        for cell in ws[1]:
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True
            )
            cell.border = cell_border

        ws.row_dimensions[1].height = 25

        # -----------------------------------------------------
        # Process each reference
        # -----------------------------------------------------

        for reference in references:
            original = reference.get("original", "")
            edited = reference.get("edited", "")

            edits = reference.get("edits") or {}

            edit_rows = []

            for item in edits.get("text_edits") or []:
                edit_rows.append(("Text Edit", item))

            for item in edits.get("missing_components") or []:
                edit_rows.append(("Missing Component", item))

            for item in edits.get("warnings") or []:
                edit_rows.append(("Warning", item))

    # Ensure every reference appears at least once.
            if not edit_rows:
                edit_rows.append(("", ""))

    # Write Original and Edited only in the first row.
            for index, (edit_type, details) in enumerate(edit_rows):
                ws.append([
                    original if index == 0 else "",
                    edited if index == 0 else "",
                    edit_type,
                    details
                ])

    # Add an empty row between references.
            ws.append([])
        # -----------------------------------------------------
        # Worksheet formatting
        # -----------------------------------------------------

        column_widths = {
            "A": 55,  # Original
            "B": 55,  # Edited
            "C": 23,  # Edit Type
            "D": 75,  # Details
        }

        for column, width in column_widths.items():
            ws.column_dimensions[column].width = width

        ws.freeze_panes = "A2"
        ws.auto_filter.ref = f"A1:D{ws.max_row}"

        # Do not print excessive empty rows in the worksheet.
        # ws.sheet_view.showGridLines = False

    # ---------------------------------------------------------
    # Save workbook
    # ---------------------------------------------------------

    output_path = Path(output_path)
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    wb.save(output_path)

    return str(output_path)

In [ ]:
create_reference_edits_excel(
    final_ref_map,
    "final_reference_results.xlsx",
)

In [ ]:
reference_create_excel.export_reference_edits_to_excel(final_ref_map)

In [ ]:
from doi_cache_helper import OpenAlexCache
cache = OpenAlexCache("cache/openalex_cache.db")
cache.get('10.1177/02698811231225609')

In [ ]:
import re
from typing import Dict


YEAR_RE = r"(?:18|19|20|21)\d{2}"


def parse_numbers_part(numbers: str) -> Dict[str, str]:
    """
    Parse the bibliographic numbers portion of a scientific reference.
    """

    result = {
        "year": "",
        "volume": "",
        "issue": "",
        "pages": "",
        "article_number": "",
    }

    if not numbers:
        return result

    text = numbers.strip()

    # ---------------------------------------------------------
    # Normalize whitespace
    # ---------------------------------------------------------

    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)

    # Normalize Unicode dash characters
    text = re.sub(r"[‐-‒–—−]", "-", text)

    # Remove trailing punctuation
    text = text.strip(" .;:")

    # ---------------------------------------------------------
    # YEAR
    # ---------------------------------------------------------

    year_match = re.search(
        rf"\b(?P<year>{YEAR_RE})\b",
        text,
        flags=re.IGNORECASE,
    )

    if not year_match:
        return result

    result["year"] = year_match.group("year")

    # Everything after year
    remainder = text[year_match.end():].strip()

    # Remove separator between year and volume
    remainder = remainder.lstrip(" ;,.")

    # ---------------------------------------------------------
    # VOLUME
    # ---------------------------------------------------------

    volume_match = re.match(
        r"(?P<volume>\d+[A-Za-z]*)",
        remainder,
        flags=re.IGNORECASE,
    )

    if not volume_match:
        return result

    result["volume"] = volume_match.group("volume")

    remainder = remainder[volume_match.end():].strip()

    # ---------------------------------------------------------
    # ISSUE / SUPPLEMENT
    #
    # Important:
    #
    # 29(3)(suppl 9):9-16
    #
    # contains TWO parenthesized components.
    #
    # Collect all consecutive (...) groups.
    # ---------------------------------------------------------

    issue_parts = []

    while remainder.startswith("("):

        # Find the matching closing parenthesis.
        #
        # This also handles nested parentheses reasonably safely.
        depth = 0
        closing_index = None

        for i, char in enumerate(remainder):
            if char == "(":
                depth += 1

            elif char == ")":
                depth -= 1

                if depth == 0:
                    closing_index = i
                    break

        if closing_index is None:
            break

        part = remainder[1:closing_index].strip()

        if part:
            issue_parts.append(part)

        remainder = remainder[closing_index + 1:].strip()

    # Combine issue components.
    #
    # (3)(suppl 9)
    # ->
    # "3 (suppl 9)"
    #
    if issue_parts:
        result["issue"] = " ".join(issue_parts)

    # ---------------------------------------------------------
    # PAGE / ARTICLE separator
    # ---------------------------------------------------------

    remainder = remainder.lstrip(" :;,.")

    if not remainder:
        return result

    location = remainder.strip(" .;,")

    # ---------------------------------------------------------
    # PAGE RANGE
    #
    # 9-16
    # 1039-1057
    # S10-S15
    # e123-e130
    # ---------------------------------------------------------

    page_range_match = re.fullmatch(
        r"""
        (?P<start>[A-Za-z]?\d+)
        \s*-\s*
        (?P<end>[A-Za-z]?\d+)
        """,
        location,
        flags=re.VERBOSE,
    )

    if page_range_match:
        result["pages"] = (
            f"{page_range_match.group('start')}-"
            f"{page_range_match.group('end')}"
        )
        return result

    # ---------------------------------------------------------
    # ARTICLE NUMBER
    #
    # e12345
    # m123
    # A123
    # ---------------------------------------------------------

    article_match = re.fullmatch(
        r"[A-Za-z]+\d+",
        location,
    )

    if article_match:
        result["article_number"] = location
        return result

    # ---------------------------------------------------------
    # SINGLE NUMERIC PAGE
    # ---------------------------------------------------------

    if re.fullmatch(r"\d+", location):
        result["pages"] = location
        return result

    # ---------------------------------------------------------
    # Unknown location.
    #
    # Preserve it rather than silently losing information.
    # ---------------------------------------------------------

    result["article_number"] = location

    return result


In [ ]:
for k,all_refs in res_list.items():
    print(k)
    for i,ref in enumerate(all_refs):
        if not ref['doi']:
            print('No DOI skipping')
            continue
        if not ref['numbers']:
            print(f'{ref['doi']}-----------{ref['numbers']}')
            print('No numbers found, skipping')
            continue
        print(f'{ref['numbers']}')
        bib_dict = parse_numbers_part(ref['numbers'])
        print(f'{i}---->{bib_dict['year']};{bib_dict['volume']}({bib_dict['issue']}):{bib_dict['pages']}')

In [ ]:
export_reference_edits_to_excel(list_for_xl)

In [ ]:
import json

final_ref = []
for j in all_ref_objs:
    final_ref.append(j.dict())

type(final_ref[0])

In [ ]:
import csv

filename = "output.csv"
with open(filename, mode="w", newline="", encoding="utf-8") as file:
    # Use the keys of the first object as the header
    writer = csv.DictWriter(file, fieldnames=final_ref[0].keys())
    
    writer.writeheader()
    writer.writerows(final_ref)



In [ ]:
from typing import List, Type, TypeVar
from pydantic import BaseModel, Field
from ollama import chat
from json_repair import repair_json

T = TypeVar("T", bound=BaseModel)

def extract_json_objects(text: str):
    objects = []
    depth = 0
    start = None
    in_string = False
    escape = False

    for i, ch in enumerate(text):
        if escape:
            escape = False
            continue

        if ch == "\\":
            escape = True
            continue

        if ch == '"':
            in_string = not in_string
            continue

        if in_string:
            continue

        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start is not None:
                objects.append(text[start:i + 1])

    return objects


def parse_repaired_objects(response: str, model: Type[T]) -> list[T]:
    result = []
    # print(response)
    for obj in extract_json_objects(response):
        repaired = repair_json(obj)
        parsed = json.loads(repaired)
        result.append(model.model_validate(parsed))

    return result

class ReferenceStructure(BaseModel):
    authors_string:str = Field(
        default="",
        description="Author names' string exactly as represented in the reference"
    )
    title: str = Field(
        default="",
        description="Full article title"
    )
    doi_string: str = Field(
        default="",
        description="DOI of the article"
    )
    extra: str = Field(
        default="",
        description="Any remaining bibliographic information not represented by the other fields"
    )


def structure_reference(reference: str) -> ReferenceStructure:
    """
    Structure a scientific reference using Ollama + Qwen3.5:4b.

    Args:
        reference: Raw reference text.

    Returns:
        ReferenceStructure Pydantic object.
    """

    prompt = f"""
You are an expert scientific reference parser.
Given a string of text from the reference list used by a scientific article,
Extract exactly four parts from the reference text: 1. authors_string 2. title 3. doi_string 4. extra.
Keep the extracted text exactly as it appears in the input. 
author_list: The complete author portion of the reference. Keep all authors and "et al." as one string. 
title: The complete article title. 
doi: The DOI string as it appears in the input. If there is no DOI, use an empty string. 
extra: Everything else in the reference that is not author_list, title, or doi. This normally contains journal, year, volume, issue, pages, PMID, etc. 
Do not rewrite, summarize, correct, or invent anything.
## Output format
Return **one** machine-readable JSON array, no extra spaces, markdown characters or line-breaks outside string literals**.
Structure of each sub-section should be like:
{{"authors_string": "Text", "title": "Text", "doi_string": "Text", "extra":"Text"}}
"""

    user_prompt = f"""{reference}"""
    response = chat(
        model="qwen3.5:4b",
        messages=[
            {
                "role": "system",
                "content": prompt,
            },
            {"role": "assistant", "content": "Okay"},
            {"role": "user", "content": user_prompt},
        ],
        format=ReferenceStructure.model_json_schema(),
        think=False,
        options={
            "temperature": 0,
        },
    )
    r = response.message.content
    print(f"response from qwen: \n{r}")
    
    res = parse_repaired_objects(r, ReferenceStructure)
    
    return res

In [ ]:
#REFERENCE EXTRACTION USING regex --- DETERMINISTIC APPROACH

In [ ]:
import csv

ref_deter = []
for k, ref in references.items():
    all_refs = ref.split('\n')[1:]
    for r in all_refs:
        ref_deter.append(parse_reference_deter(r))
    

filename = "output_deter.csv"
with open(filename, mode="w", newline="", encoding="utf-8") as file:
    # Use the keys of the first object as the header
    writer = csv.DictWriter(file, fieldnames=ref_deter[0].keys())
    
    writer.writeheader()
    writer.writerows(ref_deter)

